# From Model Scores to Governed Decisions: Complete Colab LabThis lab is the executable companion notebook for the article series **From Model Scores to Governed Decisions**.Last updated: 2026-05-16 21:25:35 +05:30It walks through the complete end-to-end workflow:1. Connect binary classification scores to real business decisions.2. Load validation labels, model scores, and segment context.3. Convert model scores into decisions using thresholds.4. Evaluate baseline model behavior.5. Review probability calibration before treating scores as policy inputs.6. Sweep candidate thresholds.7. Add business value assumptions.8. Add operational capacity constraints.9. Select a governed threshold with guardrails.10. Compare default and selected thresholds.11. Visualize metric, business-value, calibration, ROC, and precision-recall tradeoffs.12. Apply segment-specific thresholds.13. Create auditable production decision logs.14. Generate stakeholder explanations and anti-pattern checks.15. Run verification commands against the package.The goal is to show that a binary classification model does not create a business decision by itself. The threshold does, and the threshold must be governed as an operating policy.

## 0. Colab SetupRun this cell first in Google Colab. It clones the repository if needed, installs the package in editable mode, and moves into the repo folder.If you are running this notebook locally from the repository root, the same cell will use the current checkout.

In [ ]:
import osimport sysfrom pathlib import PathREPO_URL = 'https://github.com/shalabhdixit/from-model-scores-to-governed-decisions.git'REPO_DIR = Path('/content/from-model-scores-to-governed-decisions')LOCAL_MARKER = Path('pyproject.toml')if 'google.colab' in sys.modules:    if not REPO_DIR.exists():        !git clone {REPO_URL} {REPO_DIR}    os.chdir(REPO_DIR)elif LOCAL_MARKER.exists():    print('Using local repository checkout:', Path.cwd())else:    print('Run this notebook from the repository root, or use Google Colab so the repo can be cloned automatically.')!python -m pip install --quiet --upgrade pip!python -m pip install --quiet -e .[dev]print('Repository ready at:', Path.cwd())

## 1. Import The Reusable PackageThe repository is structured as a real Python package. The notebook uses the same reusable modules that the scripts and tests use.

In [ ]:
import pandas as pdfrom IPython.display import Image, displayfrom governed_decisions.anti_patterns import anti_pattern_tablefrom governed_decisions.article_concepts import binary_classification_use_cases, decision_boundary_fit_matrixfrom governed_decisions.business_value import DEFAULT_BUSINESS_VALUESfrom governed_decisions.calibration import calibration_summary, expected_calibration_error, score_distributionfrom governed_decisions.decision_logging import score_decisionfrom governed_decisions.metrics import evaluate_thresholdfrom governed_decisions.operating_curves import operating_curve_tablefrom governed_decisions.reporting import compare_thresholds, select_threshold, threshold_evaluation_reportfrom governed_decisions.sample_data import load_sample_validation_data, sample_arraysfrom governed_decisions.segment_thresholds import apply_segment_thresholdsfrom governed_decisions.thresholding import apply_thresholdfrom governed_decisions.visualization import (    plot_business_value,    plot_calibration_reliability,    plot_metric_tradeoffs,    plot_roc_precision_recall_operating_points,)pd.set_option('display.max_columns', 30)pd.set_option('display.width', 140)

## 2. Map The Article Concepts To Business DecisionsBefore touching thresholds, confirm that the business problem really is a decision-boundary problem. This section maps common binary classification use cases to threshold-driven actions and highlights when threshold tuning is useful versus when another evaluation approach is needed.

In [ ]:
print('Binary classification business contexts')display(binary_classification_use_cases())print('When threshold tuning is the right tool')display(decision_boundary_fit_matrix())

## 3. Load Validation DataA threshold can only be evaluated when you have observed outcomes and model probability scores. This sample dataset uses three fields:| Column | Meaning || --- | --- || `actual` | Observed binary outcome. `1` is the positive class. || `score` | Model probability score for the positive class. || `segment` | Optional segment used for contextual thresholds. |

In [ ]:
data = load_sample_validation_data()actual_labels, model_scores = sample_arrays()print(data.head())print('Records:', len(data))

## 4. Apply A Baseline ThresholdThe default `0.50` threshold is common, but it is only a starting point. It converts scores into class labels.

In [ ]:
baseline_threshold = 0.50data['prediction_at_050'] = apply_threshold(data['score'], threshold=baseline_threshold)data[['actual', 'score', 'prediction_at_050']].head(10)

## 5. Evaluate The Baseline ThresholdThis step computes the metrics that matter for the starting point: confusion matrix counts, precision, recall, F1, accuracy, predicted positive rate, and ROC-AUC.

In [ ]:
baseline_metrics = evaluate_threshold(actual_labels, model_scores, threshold=baseline_threshold)pd.Series(baseline_metrics)

## 6. Review Probability CalibrationThresholds only make sense when the score has a reliable meaning. Calibration review checks whether a score band such as `0.70` behaves like a roughly 70 percent likelihood in validation data. A model can rank cases well and still have probability scores that need calibration before policy approval.

In [ ]:
calibration_table = calibration_summary(actual_labels, model_scores, n_bins=5)score_band_table = score_distribution(model_scores, n_bins=5)ece = expected_calibration_error(actual_labels, model_scores, n_bins=5)display(calibration_table.round(3))display(score_band_table)print(f'Expected calibration error: {ece:.3f}')

In [ ]:
calibration_chart = plot_calibration_reliability(actual_labels, model_scores, 'outputs/calibration_reliability.png', n_bins=5)display(Image(filename=str(calibration_chart)))

## 7. Sweep Candidate ThresholdsA real decision policy should compare several candidate operating points. This report evaluates thresholds from `0.10` to `0.90`.

In [ ]:
report_without_capacity = threshold_evaluation_report(actual_labels, model_scores)report_without_capacity[[    'threshold', 'accuracy', 'precision', 'recall', 'f1',    'true_positives', 'false_positives', 'false_negatives', 'predicted_positive_rate',]]

## 8. Add Business ValueTechnical metrics count errors. Business value estimates the consequence of those errors. The default assumptions are intentionally simple and should be replaced with real finance, risk, product, and operations inputs in production.

In [ ]:
DEFAULT_BUSINESS_VALUES

In [ ]:
report_without_capacity.sort_values('business_value', ascending=False)[[    'threshold', 'precision', 'recall', 'f1', 'false_positives', 'false_negatives', 'business_value']].head(8)

## 9. Add Operational CapacityMany classification systems route cases to a human team. A threshold that creates too much review volume may be statistically attractive but operationally unusable.

In [ ]:
max_manual_reviews = 12report = threshold_evaluation_report(actual_labels, model_scores, max_manual_reviews=max_manual_reviews)report[[    'threshold', 'precision', 'recall', 'f1', 'manual_reviews', 'within_capacity', 'business_value']]

## 10. Select The Threshold With GuardrailsThe selected threshold must satisfy business and operating guardrails. In this lab, the threshold must meet minimum recall, minimum precision, and manual review capacity constraints.

In [ ]:
selected = select_threshold(    report,    min_recall=0.70,    min_precision=0.60,    require_capacity=True,)selected[['threshold', 'precision', 'recall', 'f1', 'manual_reviews', 'within_capacity', 'business_value']]

## 11. Compare Default And Selected ThresholdsStakeholders need to understand what changes when the operating threshold changes. This comparison makes the tradeoff explicit.

In [ ]:
comparison = compare_thresholds(report, baseline_threshold=0.50, selected_threshold=float(selected['threshold']))comparison

## 12. Visualize Threshold TradeoffsThe charts show two different conversations: model behavior and business consequence.

In [ ]:
metric_chart = plot_metric_tradeoffs(report, 'outputs/metric-tradeoffs.png')business_chart = plot_business_value(report, 'outputs/business-value.png', selected_threshold=float(selected['threshold']))display(Image(filename=str(metric_chart)))display(Image(filename=str(business_chart)))

## 13. Compare ROC And Precision-Recall Operating ViewsROC-AUC describes ranking quality across thresholds. Precision-recall shows what the selected threshold means for positive-case workload. Enterprise teams need both views because a threshold can look strong statistically while still creating costly or low-quality action queues.

In [ ]:
operating_view = operating_curve_table(actual_labels, model_scores, operating_threshold=float(selected['threshold']))display(operating_view.round(3))roc_pr_chart = plot_roc_precision_recall_operating_points(    actual_labels,    model_scores,    'outputs/roc_precision_recall_operating_point.png',    operating_threshold=float(selected['threshold']),)display(Image(filename=str(roc_pr_chart)))

## 14. Apply Segment-Specific ThresholdsSegment-aware thresholds can improve operational fit, but they also create governance obligations. In production, each segment threshold should have rationale, approval, monitoring, and fairness review.

In [ ]:
segment_thresholds = {    'high_value': 0.65,    'standard': 0.45,}segmented_data = apply_segment_thresholds(data, segment_thresholds)segmented_data[['actual', 'score', 'segment', 'segment_threshold', 'segment_prediction']].head(12)

## 15. Create A Production Decision RecordA production decision record should preserve the score, threshold, predicted label, route, model version, threshold version, and timestamp. This is the foundation for auditability and rollback.

In [ ]:
decision = score_decision(    record_id='TXN-10001',    model_score=0.62,    threshold=float(selected['threshold']),    model_version='fraud-model-v4',    threshold_version='threshold-policy-2026-05',)decision

## 16. Explain The Threshold To StakeholdersA governed threshold needs a business explanation. This cell turns the comparison table into plain language that a product, risk, operations, or compliance owner can review.

In [ ]:
baseline_row = comparison.loc[comparison['threshold'] == baseline_threshold].iloc[0]selected_row = comparison.loc[comparison['threshold'] == float(selected['threshold'])].iloc[0]stakeholder_explanation = (    f"We are recommending a {selected_row['threshold']:.2f} operating threshold instead of the default "    f"{baseline_row['threshold']:.2f}. It preserves recall at {selected_row['recall']:.2f}, improves precision "    f"from {baseline_row['precision']:.2f} to {selected_row['precision']:.2f}, keeps manual reviews at "    f"{int(selected_row['manual_reviews'])}, and increases expected business value from "    f"{baseline_row['business_value']:.0f} to {selected_row['business_value']:.0f}. This threshold should be "    "approved, versioned, monitored, and governed as a business policy.")print(stakeholder_explanation)

## 17. Review Common Threshold Anti-PatternsThe last technical step is to check the policy against common production failure modes. These are the patterns that usually make threshold projects look complete in notebooks but weak in operations.

In [ ]:
display(anti_pattern_table())

## 18. Governance ChecklistBefore a threshold becomes production policy, the team should answer these questions:| Governance Question | Why It Matters || --- | --- || Which model version produced the score? | Supports model lineage and incident investigation. || Which threshold version was active? | Supports policy audit and rollback. || What business value assumptions were used? | Prevents hidden cost and risk assumptions. || Did the policy fit operational capacity? | Prevents queue overload and SLA failure. || Did calibration support policy approval? | Prevents unreliable probability scores from becoming operating rules. || Did ROC and precision-recall views support the operating point? | Prevents ranking metrics from hiding workload quality. || Were segment thresholds reviewed for fairness? | Prevents unmanaged disparate impact. || Who approved the threshold? | Makes ownership explicit. || How will drift and overrides be monitored? | Keeps the decision policy healthy after deployment. |

## 19. Run Package VerificationThis final cell runs the same validation commands used by the repository. In Colab this confirms that the notebook, examples, reusable modules, and tests remain aligned.

In [ ]:
!python -m pytest!python examples/00_article_concept_map.py!python examples/03b_probability_calibration_review.py!python examples/08b_roc_vs_precision_recall.py!python examples/14_common_anti_patterns.py!python examples/15_stakeholder_explanation.py!python examples/complete_working_example.py

## 20. Final TakeawayThe model estimates probability. The threshold defines behavior. Governance determines whether that behavior can be trusted.A complete production workflow does more than find a better metric. It validates score reliability, simulates business value, checks capacity, compares operating curves, explains the policy to stakeholders, preserves decision evidence, and monitors the threshold as a governed business asset.